# AeroRAG-X grounded-agent GRPO on a Kaggle P100
This notebook first proves the GPU training path with synthetic fixtures. Point `CASES_PATH` at a real, versioned, protected-disjoint JSONL before treating a run as experiment evidence.

In [ ]:
!nvidia-smi
import subprocess
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
print('GPU:', gpu)
assert 'P100' in gpu, 'Select GPU P100 in Kaggle notebook settings before continuing.'

In [ ]:
from pathlib import Path
import os
ROOT = Path('/kaggle/working/AeroRAG-X')
if not ROOT.exists():
    os.system('git clone https://github.com/triasha72/AeroRAG-X.git /kaggle/working/AeroRAG-X')
os.chdir(ROOT)
!python -m pip install -q -e '.[llm,rl]'


In [ ]:
CASES_PATH = 'data/training/grpo_grounded_agent_v0_1.template.jsonl'
CONFIG_PATH = 'configs/grpo_kaggle_p100_smoke_v0_1.yaml'
OUTPUT_DIR = Path('/kaggle/working/aeroragx-grpo')
if CASES_PATH.endswith('.template.jsonl'):
    print('SMOKE TEST ONLY: these cases cannot establish model improvement.')
!python scripts/train_grpo_grounded_agent_v0_1.py --cases {CASES_PATH} --config {CONFIG_PATH} --output-dir {OUTPUT_DIR}


In [ ]:
# Remove --execute for validation only. Add --resume after an interrupted session.
!python scripts/train_grpo_grounded_agent_v0_1.py --cases {CASES_PATH} --config {CONFIG_PATH} --output-dir {OUTPUT_DIR} --execute --resume


In [ ]:
import hashlib, json, tarfile
receipt = json.loads((OUTPUT_DIR / 'run_receipt.json').read_text())
assert (OUTPUT_DIR / 'final_adapter').is_dir()
archive = Path('/kaggle/working/aeroragx-grpo-output.tar.gz')
with tarfile.open(archive, 'w:gz') as tar:
    tar.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name)
print(json.dumps(receipt, indent=2))
print('archive_sha256:', hashlib.sha256(archive.read_bytes()).hexdigest())
